# KidneyVision AI — Stage 2 Diagnostic Model & Grad-CAM Engine
### High-Accuracy Kidney Stone Detection (Normal vs. Stone) with Explainable AI

This notebook trains the **Stage 2 Diagnostic Big Model** for KidneyVision AI and implements the **Grad-CAM (Gradient-weighted Class Activation Mapping) Engine** to localize nephrolithiasis (kidney stones).

#### Clinical & Technical Purpose:
- **Diagnostic Classification:** High-sensitivity, high-specificity binary classification between **Normal** (healthy renal parenchyma) and **Stone** (presence of calculus/nephrolithiasis).
- **Explainable AI (XAI):** Mathematically traces the deep neural network gradients back to the final convolutional feature maps (`conv5_block3_out`), generating a spatial activation heatmap.
- **Stone Pinpointing:** Identifies peak activation coordinates $(X\%, Y\%)$ to render precise target pins in the clinical user interface.

#### Architecture:
- **Backbone:** ResNet50V2 (Pretrained on ImageNet)
- **Target Conv Layer:** `conv5_block3_out` ($7 \times 7 \times 2048$ spatial feature maps)
- **Classification Head:** GlobalAveragePooling2D $\rightarrow$ BatchNorm $\rightarrow$ Dropout(0.4) $\rightarrow$ Dense(128, ReLU) $\rightarrow$ BatchNorm $\rightarrow$ Dropout(0.2) $\rightarrow$ Dense(1, Sigmoid)
- **Export Target:** `kidney_stone_diagnostic_model.keras`

In [ ]:
import os
import sys
import io
import glob
import base64
import hashlib
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, metrics

# Reproducibility Seed
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"Available GPUs: {gpus if gpus else 'None (Running on CPU)'}")


## 1. Dataset Paths & Configuration

Configure your diagnostic dataset paths. The dataset must contain confirmed **Kidney Scans** divided into `Normal` and `Stone`:
```
dataset_diagnostic/
├── train/
│   ├── Normal/
│   └── Stone/
└── val/
    ├── Normal/
    └── Stone/
```

In [ ]:
# ==========================================
# HYPERPARAMETERS & CONFIGURATION
# ==========================================
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
BATCH_SIZE = 32

# Paths (Update to match your local or Kaggle directories)
DATASET_DIR = "dataset_diagnostic"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "val")
TEST_DIR = os.path.join(DATASET_DIR, "test")

# Target output file names
MODEL_SAVE_PATH = "kidney_stone_diagnostic_model.keras"
LAST_CONV_LAYER_NAME = "conv5_block3_out"

print(f"Model will be saved to: {MODEL_SAVE_PATH}")
print(f"Grad-CAM Target Convolutional Layer: {LAST_CONV_LAYER_NAME}")


## 2. Dataset Cleaning & Integrity Audit

Scans all images before training to eliminate:
1. Corrupted/unreadable image files
2. Zero-byte files
3. Duplicate images via MD5 hashing to prevent data leakage between splits

In [ ]:
def clean_and_audit_dataset(directory_path: str):
    """Scans directory to remove unreadable, zero-byte, and duplicate images."""
    if not os.path.exists(directory_path):
        print(f"[WARNING] Directory not found: {directory_path}. Skipping audit.")
        return

    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".jfif")
    seen_hashes = set()
    removed_corrupted = 0
    removed_duplicates = 0
    total_valid = 0

    print(f"\n[DATA AUDIT] Auditing images in: {directory_path}...")
    
    for root, _, files in os.walk(directory_path):
        for fname in files:
            fpath = os.path.join(root, fname)
            
            if not fname.lower().endswith(valid_extensions):
                continue

            # Check file size > 0
            if os.path.getsize(fpath) == 0:
                try:
                    os.remove(fpath)
                    removed_corrupted += 1
                except OSError:
                    pass
                continue

            # Verify image format integrity
            try:
                with Image.open(fpath) as img:
                    img.verify()
                
                # MD5 hash check for duplicate removal
                with open(fpath, "rb") as f:
                    file_hash = hashlib.md5(f.read()).hexdigest()
                    
                if file_hash in seen_hashes:
                    try:
                        os.remove(fpath)
                        removed_duplicates += 1
                    except OSError:
                        pass
                else:
                    seen_hashes.add(file_hash)
                    total_valid += 1

            except Exception:
                try:
                    os.remove(fpath)
                    removed_corrupted += 1
                except OSError:
                    pass

    print(f"Audit Complete for {directory_path}:")
    print(f"  - Valid unique images: {total_valid}")
    print(f"  - Corrupted files removed: {removed_corrupted}")
    print(f"  - Duplicate files removed: {removed_duplicates}")

# Execute audit on diagnostic splits
if os.path.exists(TRAIN_DIR):
    clean_and_audit_dataset(TRAIN_DIR)
if os.path.exists(VAL_DIR):
    clean_and_audit_dataset(VAL_DIR)


## 3. High-Performance tf.data Input Pipeline & Clinical Augmentations

Applies medical-specific data augmentations:
- Horizontal flip (kidneys are bilaterally symmetric)
- Subtle rotation (patient angle variance)
- Contrast/brightness jitter (simulating ultrasound probe gain and CT windowing variations)
- ResNet50V2 normalization (`preprocess_input`)

In [ ]:
if not os.path.exists(TRAIN_DIR) or not os.path.exists(VAL_DIR):
    print("[NOTE] Diagnostic dataset folders not found at specified paths yet.")
    print(f"Please configure TRAIN_DIR ({TRAIN_DIR}) and VAL_DIR ({VAL_DIR}) when ready to run.")
else:
    print("Loading Diagnostic Train Dataset...")
    train_raw = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="binary",
        shuffle=True,
        seed=SEED
    )

    print("Loading Diagnostic Validation Dataset...")
    val_raw = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="binary",
        shuffle=False
    )

    class_names = train_raw.class_names
    print(f"Classes detected: {class_names}")
    # Expected: ['Normal', 'Stone'] -> Normal=0, Stone=1

    data_augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.06),
        layers.RandomZoom(0.06),
        layers.RandomContrast(0.12),
        layers.RandomBrightness(0.08)
    ], name="medical_augmentation")

    AUTOTUNE = tf.data.AUTOTUNE

    def prepare_train_ds(ds):
        return (
            ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
              .shuffle(buffer_size=1000, seed=SEED)
              .prefetch(buffer_size=AUTOTUNE)
        )

    def prepare_val_ds(ds):
        return ds.prefetch(buffer_size=AUTOTUNE)

    train_dataset = prepare_train_ds(train_raw)
    val_dataset = prepare_val_ds(val_raw)
    print("tf.data pipelines prepared successfully.")


## 4. Architecture Definition: ResNet50V2 + Grad-CAM Optimized Head

We use **ResNet50V2** as the feature extractor:
- Residual connections prevent vanishing gradients when computing Grad-CAM backpropagation.
- Feature maps from `conv5_block3_out` preserve sharp, hyperdense calcification patterns.
- **Global Average Pooling** replaces the fragile 11M-parameter Flatten layer, creating a direct mathematical link between conv features and the Stone output probability.

In [ ]:
def build_diagnostic_model(input_shape=(224, 224, 3)):
    # 1. Pretrained Backbone
    base_model = tf.keras.applications.ResNet50V2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False  # Frozen for Phase 1

    # 2. Input Layer
    inputs = tf.keras.Input(shape=input_shape, name="scan_input")
    
    # ResNet50V2 normalization: scales [-1, 1]
    x = tf.keras.applications.resnet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    
    # 3. Clinical Diagnostic Head
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="diag_bn1")(x)
    x = layers.Dropout(0.4, name="diag_dropout1")(x)
    x = layers.Dense(128, activation="relu", name="diag_dense1")(x)
    x = layers.BatchNormalization(name="diag_bn2")(x)
    x = layers.Dropout(0.2, name="diag_dropout2")(x)
    
    # Sigmoid output (0 = Normal, 1 = Stone)
    outputs = layers.Dense(1, activation="sigmoid", name="stone_probability")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="KidneyVision_ResNet50V2_Diagnostic")
    return model, base_model

diag_model, base_backbone = build_diagnostic_model()
diag_model.summary()


## 5. Phase 1: Feature Extraction Training (Backbone Frozen)

Trains the classification head while keeping the ResNet50V2 backbone weights frozen.

In [ ]:
diag_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        metrics.Precision(name="precision"),
        metrics.Recall(name="recall"),
        metrics.AUC(name="auc")
    ]
)

callbacks_phase1 = [
    callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

if "train_dataset" in locals() and "val_dataset" in locals():
    print("Starting Phase 1 Training (Diagnostic Head)...")
    history_p1 = diag_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=8,
        callbacks=callbacks_phase1
    )
else:
    print("[READY] Run after loading datasets to execute Phase 1.")


## 6. Phase 2: Fine-Tuning Top Residual Blocks

Unfreezes the top 40 layers of ResNet50V2 (`conv5_block1` through `conv5_block3`) with a low learning rate ($10^{-5}$) to adapt high-level convolutional kernels to renal calculi features.

In [ ]:
# Unfreeze top layers of ResNet50V2
base_backbone.trainable = True
for layer in base_backbone.layers[:-40]:
    layer.trainable = False

diag_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        metrics.Precision(name="precision"),
        metrics.Recall(name="recall"),
        metrics.AUC(name="auc")
    ]
)

callbacks_phase2 = [
    callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

if "train_dataset" in locals() and "val_dataset" in locals():
    print("Starting Phase 2 Fine-Tuning...")
    history_p2 = diag_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=15,
        callbacks=callbacks_phase2
    )
else:
    print("[READY] Run after Phase 1.")


## 7. Model Evaluation & Clinical Triage Validation

Evaluates the diagnostic model on the validation set:
- Confusion Matrix (Normal vs. Stone)
- Precision, Sensitivity (Recall), Specificity, F1-Score
- ROC-AUC Curve
- Uncertainty Zone analysis ($[0.35, 0.65]$ triage boundary)

In [ ]:
if "val_dataset" in locals():
    from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
    
    y_true = []
    y_pred_probs = []

    for images, labels in val_dataset:
        probs = diag_model.predict(images, verbose=0)
        y_true.extend(labels.numpy().flatten())
        y_pred_probs.extend(probs.flatten())

    y_true = np.array(y_true)
    y_pred_probs = np.array(y_pred_probs)

    threshold = 0.50
    y_pred = (y_pred_probs >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    print("\n" + "="*50)
    print(f"CONFUSION MATRIX (Diagnostic Threshold = {threshold}):")
    print(cm)
    print("="*50)

    print("\nCLASSIFICATION REPORT:")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Stone"]))

    auc_val = roc_auc_score(y_true, y_pred_probs)
    print(f"Validation ROC-AUC: {auc_val:.4f}")
    
    # Clinical uncertainty audit
    uncertain_mask = (y_pred_probs >= 0.35) & (y_pred_probs <= 0.65)
    uncertain_count = np.sum(uncertain_mask)
    print(f"Borderline / Uncertain cases [0.35 - 0.65]: {uncertain_count} ({uncertain_count / len(y_true) * 100:.2f}%)")
else:
    print("[READY] Run after training to evaluate metrics.")


## 8. The Grad-CAM Engine: Stone Localization & Peak Coordinates

Computes Grad-CAM using `conv5_block3_out`:
1. Extracts convolutional feature maps $A^k$ and gradients $\frac{\partial y^c}{\partial A^k}$.
2. Computes channel importance weights $\alpha^k = \frac{1}{Z}\sum_i \sum_j \frac{\partial y^c}{\partial A^k_{i,j}}$.
3. Computes weighted activation map: $L_{\text{Grad-CAM}} = \text{ReLU}\left(\sum_k \alpha^k A^k\right)$.
4. Finds peak coordinates $(X\%, Y\%)$ of maximum activation to point directly to the stone.
5. Renders a Jet colormap with transparent background ($< 15\%$ activation) so non-stone kidney tissue remains clearly visible.

In [ ]:
def compute_gradcam_resnet(model, img_batch, target_layer_name="conv5_block3_out"):
    """
    Universal Grad-CAM engine for ResNet50V2 diagnostic model.
    Returns (gradcam_base64, peak_coordinates, heatmap_array).
    """
    # 1. Multi-output gradient model
    # In ResNet50V2, the backbone is a nested functional layer or part of the model
    try:
        # Try direct layer access
        target_layer = model.get_layer(target_layer_name)
        grad_model = tf.keras.Model(
            inputs=[model.inputs],
            outputs=[target_layer.output, model.output]
        )
    except ValueError:
        # If target layer is inside the base_model
        base_layer = model.get_layer("resnet50v2")
        target_conv = base_layer.get_layer(target_layer_name)
        
        # Create intermediate submodel for base
        base_submodel = tf.keras.Model(
            inputs=base_layer.inputs,
            outputs=[target_conv.output, base_layer.output]
        )
        
        # Forward through full model
        inputs = model.inputs
        x = tf.keras.applications.resnet_v2.preprocess_input(inputs[0])
        conv_out, base_out = base_submodel(x)
        x = model.get_layer("gap")(base_out)
        x = model.get_layer("diag_bn1")(x)
        x = model.get_layer("diag_dropout1")(x)
        x = model.get_layer("diag_dense1")(x)
        x = model.get_layer("diag_bn2")(x)
        x = model.get_layer("diag_dropout2")(x)
        preds = model.get_layer("stone_probability")(x)
        grad_model = tf.keras.Model(inputs=inputs, outputs=[conv_out, preds])

    # 2. GradientTape to record operations
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch)
        stone_score = predictions[:, 0]

    # 3. Gradients of Stone score with respect to feature maps
    grads = tape.gradient(stone_score, conv_outputs)
    if grads is None or conv_outputs is None:
        return None, None, None

    # 4. Importance weights (global average of gradients across H and W)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # 5. Weighted combination of 2048 feature maps
    conv_out = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)

    # 6. ReLU to isolate positive contributions to Stone class
    heatmap = tf.maximum(heatmap, 0)
    max_act = tf.math.reduce_max(heatmap)
    if max_act > 0:
        heatmap = heatmap / max_act
    heatmap_np = heatmap.numpy()

    # 7. Peak Activation Coordinates
    peak_y, peak_x = np.unravel_index(np.argmax(heatmap_np), heatmap_np.shape)
    peak_coords = {
        "x": round(float(peak_x / heatmap_np.shape[1]) * 100, 1),
        "y": round(float(peak_y / heatmap_np.shape[0]) * 100, 1)
    }

    # 8. Upsample Heatmap to Image Size
    h_img = Image.fromarray((heatmap_np * 255).astype(np.uint8), mode="L")
    h_resized = np.array(h_img.resize((IMG_WIDTH, IMG_HEIGHT), Image.Resampling.BILINEAR), dtype=np.float32) / 255.0

    # 9. Jet Colormap RGBA with Alpha Transparency
    r = np.clip(1.5 - np.abs(4.0 * h_resized - 3.0), 0.0, 1.0)
    g = np.clip(1.5 - np.abs(4.0 * h_resized - 2.0), 0.0, 1.0)
    b = np.clip(1.5 - np.abs(4.0 * h_resized - 1.0), 0.0, 1.0)
    # Low activations (< 0.15) fade completely to transparent
    alpha = np.clip((h_resized - 0.15) / 0.85, 0.0, 1.0) * 0.85

    rgba = np.stack([r * 255, g * 255, b * 255, alpha * 255], axis=-1).astype(np.uint8)
    out_pil = Image.fromarray(rgba, mode="RGBA")

    buf = io.BytesIO()
    out_pil.save(buf, format="PNG")
    gradcam_b64 = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode('utf-8')}"

    return gradcam_b64, peak_coords, h_resized

print("Grad-CAM Engine successfully configured for ResNet50V2.")


## 9. Grad-CAM Visualizer: Side-by-Side Verification

Plots the original scan, the raw activation heatmap, and the transparent overlay with the stone target marker.

In [ ]:
def visualize_prediction_and_gradcam(image_path: str):
    """Displays the original scan side-by-side with Grad-CAM and peak crosshair."""
    if not os.path.exists(image_path):
        print(f"File not found: {image_path}")
        return

    raw_img = Image.open(image_path).convert("RGB")
    resized_img = raw_img.resize((IMG_WIDTH, IMG_HEIGHT))
    img_arr = np.array(resized_img, dtype=np.float32)
    img_batch = np.expand_dims(img_arr, axis=0)

    # Predict
    prob = float(diag_model.predict(img_batch, verbose=0)[0][0])
    prediction = "Stone" if prob >= 0.50 else "Normal"
    confidence = (prob if prediction == "Stone" else (1.0 - prob)) * 100

    # Grad-CAM
    _, peak_coords, heatmap = compute_gradcam_resnet(diag_model, img_batch)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. Original Scan
    axes[0].imshow(resized_img)
    axes[0].set_title(f"Scan Input\nPred: {prediction} ({confidence:.1f}%)")
    axes[0].axis("off")

    # 2. Grad-CAM Heatmap
    if heatmap is not None:
        axes[1].imshow(heatmap, cmap="jet")
        axes[1].set_title(f"Grad-CAM Heatmap\n(Stone Activation)")
        axes[1].axis("off")

        # 3. Superimposed with Pin
        axes[2].imshow(resized_img)
        axes[2].imshow(heatmap, cmap="jet", alpha=0.5)
        if peak_coords and prediction == "Stone":
            px = (peak_coords["x"] / 100.0) * IMG_WIDTH
            py = (peak_coords["y"] / 100.0) * IMG_HEIGHT
            axes[2].plot(px, py, "ro", markersize=10, markeredgecolor="yellow", markeredgewidth=2)
            axes[2].set_title(f"Stone Pinpoint Target\nPeak: X={peak_coords['x']}%, Y={peak_coords['y']}%")
        else:
            axes[2].set_title("No Significant Stone Focus")
        axes[2].axis("off")

    plt.tight_layout()
    plt.show()

print("Visualizer function ready.")


## 10. Complete Production-Ready Inference Function

Replicates the JSON response payload required by the Flask microservice (`ai-flask-service`).

In [ ]:
def predict_and_explain(image_path: str):
    """Complete clinical inference workflow matching Flask production requirements."""
    if not os.path.exists(image_path):
        return {"error": f"Image file not found: {image_path}"}

    raw_img = Image.open(image_path).convert("RGB")
    resized = raw_img.resize((IMG_WIDTH, IMG_HEIGHT))
    img_batch = np.expand_dims(np.array(resized, dtype=np.float32), axis=0)

    prob = float(diag_model.predict(img_batch, verbose=0)[0][0])
    class_idx = 1 if prob >= 0.50 else 0
    prediction = "Stone" if class_idx == 1 else "Normal"
    confidence = round((prob if class_idx == 1 else (1.0 - prob)) * 100, 2)

    is_uncertain = bool((0.35 <= prob <= 0.65) or (confidence < 70.0))
    triage_status = "review" if is_uncertain else "completed"

    gradcam_b64, peak_coords, _ = compute_gradcam_resnet(diag_model, img_batch)

    return {
        "task": "Kidney Stone Detection",
        "prediction": prediction,
        "confidence": confidence,
        "probability": round(prob, 4),
        "is_uncertain": is_uncertain,
        "status": triage_status,
        "gradcam_image": gradcam_b64,
        "peak_coordinates": peak_coords
    }

print("Production inference simulation ready.")


## 11. Final Model Export & Verification

Saves the trained model to `kidney_stone_diagnostic_model.keras` and verifies that it reloads with zero errors.

In [ ]:
# Save model in native Keras 3 format
diag_model.save(MODEL_SAVE_PATH)
print(f"Diagnostic model saved successfully to: {os.path.abspath(MODEL_SAVE_PATH)}")

# Reload verification
print("Verifying model reload...")
reloaded = tf.keras.models.load_model(MODEL_SAVE_PATH)
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
warmup = reloaded.predict(dummy, verbose=0)
print(f"Reload verification successful! Output shape: {warmup.shape}")
print("\n*** You can now copy kidney_stone_diagnostic_model.keras into ai-flask-service/! ***")
